In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)
print("Setup complete! Ready to build the pipeline.")

Setup complete! Ready to build the pipeline.


In [10]:
# Number of historical bids to generate
n_samples = 1000

# 1. Client Budget (in thousands of USD, e.g., $10k to $250k)
budget_k = np.random.randint(10, 250, size=n_samples)

# 2. Has a Problem to Solve (0 = Vague/No, 1 = Well-defined problem)
has_problem = np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7])
# 3. Aptitude & Ability to follow through (0 = Low, 1 = High)
has_aptitude = np.random.choice([0, 1], size=n_samples, p=[0.4, 0.6])

# Create a DataFrame
df = pd.DataFrame({
    'client_budget_k': budget_k,
    'has_problem': has_problem,
    'has_aptitude': has_aptitude
})

# Define a scoring function to determine historical success (the ground truth)
# Bids with higher budgets, a clear problem, and capable clients are much more likely to succeed
score = (df['client_budget_k'] / 50) + (df['has_problem'] * 3) + (df['has_aptitude'] * 4)

# Convert score to a probability using a sigmoid function, then add some randomness
prob_success = 1 / (1 + np.exp(-(score - 6)))
df['bid_success'] = np.random.binomial(1, prob_success)

# Preview the synthetic historical data
print(f"Generated {n_samples} historical records.")
df.head()

Generated 1000 historical records.


,client_budget_k,has_problem,has_aptitude,bid_success
0,112,0,1,0
1,189,1,0,1
2,102,1,1,1
3,24,1,1,1
4,116,0,1,1


In [11]:
# Features (X) and Target (y)
X = df[['client_budget_k', 'has_problem', 'has_aptitude']]
y = df['bid_success']

# Split into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Training set size: 800 samples
Testing set size: 200 samples


In [12]:
# Initialize Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [13]:
# Predict on test data
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1] # Probability of success (class 1)

# Print Evaluation Metrics
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.2f}")

# Feature Importance
importances = rf_model.feature_importances_
for feature, importance in zip(X.columns, importances):
    print(f"Feature '{feature}' Importance: {importance:.2f}")

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.74      0.82      0.78        71
           1       0.89      0.84      0.87       129

    accuracy                           0.83       200
   macro avg       0.82      0.83      0.82       200
weighted avg       0.84      0.83      0.84       200

ROC-AUC Score: 0.93
Feature 'client_budget_k' Importance: 0.35
Feature 'has_problem' Importance: 0.21
Feature 'has_aptitude' Importance: 0.44


In [14]:
def evaluate_new_bid(budget_k, clear_problem, client_aptitude, threshold=0.70):
    """
    Evaluates a single incoming bid pipeline opportunity.
    clear_problem: 1 for Yes, 0 for No
    client_aptitude: 1 for High, 0 for Low
    threshold: Minimum probability to accept the bid
    """
    # Format the input into a DataFrame matching the model's structure
    new_bid = pd.DataFrame([{
        'client_budget_k': budget_k,
        'has_problem': clear_problem,
        'has_aptitude': client_aptitude
    }])

    # Extract the probability of success
    prob = rf_model.predict_proba(new_bid)[0][1]

    # Decision logic
    decision = "🚀 GO (High probability of success)" if prob >= threshold else "❌ NO-GO (Pass or re-qualify)"

    print(f"--- Bid Evaluation Result ---")
    print(f"Estimated Success Probability: {prob * 100:.1f}%")
    print(f"Decision: {decision}\n")

# --- TEST SCENARIOS ---

# Scenario A: Great budget, clear problem, but client lacks aptitude/follow-through
evaluate_new_bid(budget_k=150, clear_problem=1, client_aptitude=0)

# Scenario B: Modest budget, but they have a clear problem and high aptitude
evaluate_new_bid(budget_k=60, clear_problem=1, client_aptitude=1)

# Scenario C: Massive budget, clear problem, high aptitude
evaluate_new_bid(budget_k=200, clear_problem=1, client_aptitude=1)

--- Bid Evaluation Result ---
Estimated Success Probability: 51.2%
Decision: ❌ NO-GO (Pass or re-qualify)

--- Bid Evaluation Result ---
Estimated Success Probability: 90.3%
Decision: 🚀 GO (High probability of success)

--- Bid Evaluation Result ---
Estimated Success Probability: 99.1%
Decision: 🚀 GO (High probability of success)

